In [1]:
import requests
import pandas as pd
from tqdm import tqdm
import time
import os

# 1. Data Retrieval

In [2]:
# STEP 1 — API Setup

API_KEY = 'oe_QGgWdwVTERbwcSmN5Nhwnt'

BASE_URL = "https://api.openelectricity.org.au/v4"

NETWORK = "NEM"

STATUS = "operating"

INTERVAL = "5m"

START_DATE = "2026-05-09T00:00:00"

END_DATE = "2026-05-16T00:00:00"

HEADERS = {
    "Authorization": f"Bearer {API_KEY}"
}

In [3]:
# STEP 2 — Get Facilities List

facility_url = f"{BASE_URL}/facilities/"

facility_params = {
    "network_id": NETWORK,
    "status_id": STATUS
}

facility_response = requests.get(
    facility_url,
    headers=HEADERS,
    params=facility_params
)

print("Status Code:", facility_response.status_code)


# Convert JSON

facility_json = facility_response.json()


# Extract data

facility_data = facility_json["data"]


# Check total facilities

print("Total Facilities:", len(facility_data))


# View first facility

facility_data[0]

Status Code: 200
Total Facilities: 432


{'code': 'ADP',
 'code_display': 'ADP',
 'name': 'Adelaide Desalination',
 'network_id': 'NEM',
 'network_region': 'SA1',
 'description': '<p>The Adelaide Desalination plant (ADP), formerly known as the Port Stanvac Desalination Plant, is a sea water reverse osmosis desalination plant located in Lonsdale, South Australia which has the capacity to provide the city of Adelaide with up to 50% of its drinking water needs.</p>',
 'osm_way_id': '12798948',
 'location': {'lat': -35.100751, 'lng': 138.483357},
 'units': [{'code': 'ADPPV1',
   'code_display': 'ADPPV1',
   'fueltech_id': 'solar_utility',
   'status_id': 'operating',
   'capacity_registered': 24.75,
   'capacity_maximum': 19.0,
   'data_first_seen': '2021-05-18T13:10:00+10:00',
   'data_last_seen': '2026-05-27T17:50:00+10:00',
   'dispatch_type': 'GENERATOR',
   'commencement_date': '2021-05-17T14:00:00+10:00',
   'commencement_date_specificity': 'day',
   'expected_closure_date': '2040-12-31T14:00:00+10:00',
   'expected_closure

In [4]:
# STEP 3 — Create Empty Data Dictionary

data_dict = {

    "facility_name": [],

    "facility_code": [],

    "unit_code": [],

    "power_mw": [],

    "emissions_t": [],

    "timestamp": [],

    "lat": [],

    "lon": []
}

In [5]:
# STEP 4 Exploration / Structure Inspection
first_facility = facility_data[0]

first_facility

{'code': 'ADP',
 'code_display': 'ADP',
 'name': 'Adelaide Desalination',
 'network_id': 'NEM',
 'network_region': 'SA1',
 'description': '<p>The Adelaide Desalination plant (ADP), formerly known as the Port Stanvac Desalination Plant, is a sea water reverse osmosis desalination plant located in Lonsdale, South Australia which has the capacity to provide the city of Adelaide with up to 50% of its drinking water needs.</p>',
 'osm_way_id': '12798948',
 'location': {'lat': -35.100751, 'lng': 138.483357},
 'units': [{'code': 'ADPPV1',
   'code_display': 'ADPPV1',
   'fueltech_id': 'solar_utility',
   'status_id': 'operating',
   'capacity_registered': 24.75,
   'capacity_maximum': 19.0,
   'data_first_seen': '2021-05-18T13:10:00+10:00',
   'data_last_seen': '2026-05-27T17:50:00+10:00',
   'dispatch_type': 'GENERATOR',
   'commencement_date': '2021-05-17T14:00:00+10:00',
   'commencement_date_specificity': 'day',
   'expected_closure_date': '2040-12-31T14:00:00+10:00',
   'expected_closure

In [6]:
# STEP 5 — Request Power + Emissions Data

for facility in facility_data:

    # Facility Information
    facility_code = facility["code"]

    facility_name = facility["name"]

    lat = facility["location"]["lat"]

    lon = facility["location"]["lng"]

    # API URL
    data_url = f"{BASE_URL}/data/facilities/{NETWORK}"
    
    # Request Power Data
    power_params = {
        "metrics": ["power"],
        "interval": INTERVAL,
        "facility_code": facility_code,
        "date_start": START_DATE,
        "date_end": END_DATE
    }

    power_response = requests.get(
        data_url,
        headers=HEADERS,
        params=power_params
    )

    power_json = power_response.json()
    
    # Request Emissions Data
    emissions_params = {
        "metrics": ["emissions"],
        "interval": INTERVAL,
        "facility_code": facility_code,
        "date_start": START_DATE,
        "date_end": END_DATE
    }

    emissions_response = requests.get(
        data_url,
        headers=HEADERS,
        params=emissions_params
    )

    emissions_json = emissions_response.json()
  
    # STEP 6 — Extract Power + Emissions
    
    # Error Handling
    error_codes = [400, 401, 403, 404, 416, 422, 500]

    if power_response.status_code in error_codes:
        continue

    if emissions_response.status_code in error_codes:
        continue

    power_data = power_json["data"][0]["results"][0]

    emissions_data = emissions_json["data"][0]["results"][0]

    
    # STEP 7 — Integrate Facility Data
    
    for power_row, emissions_row in zip(
        power_data["data"],
        emissions_data["data"]
    ):

        timestamp = power_row[0]

        power_value = power_row[1]

        emissions_value = emissions_row[1]

        data_dict["facility_name"].append(
            facility_name
        )

        data_dict["facility_code"].append(
            facility_code
        )

        data_dict["unit_code"].append(
            power_data["columns"]["unit_code"]
        )

        data_dict["power_mw"].append(
            power_value
        )

        data_dict["emissions_t"].append(
            emissions_value
        )

        data_dict["timestamp"].append(
            timestamp
        )

        data_dict["lat"].append(
            lat
        )

        data_dict["lon"].append(
            lon
        )

In [7]:
# STEP 8 — Create DataFrame
df = pd.DataFrame(data_dict)

df.head()

,facility_name,facility_code,unit_code,power_mw,emissions_t,timestamp,lat,lon
0,Adelaide Desalination,ADP,ADPBA1,-0.005,0.0,2026-05-09T00:00:00+10:00,-35.100751,138.483357
1,Adelaide Desalination,ADP,ADPBA1,0.027,0.0,2026-05-09T00:05:00+10:00,-35.100751,138.483357
2,Adelaide Desalination,ADP,ADPBA1,0.008,0.0,2026-05-09T00:10:00+10:00,-35.100751,138.483357
3,Adelaide Desalination,ADP,ADPBA1,0.071,0.0,2026-05-09T00:15:00+10:00,-35.100751,138.483357
4,Adelaide Desalination,ADP,ADPBA1,0.009,0.0,2026-05-09T00:20:00+10:00,-35.100751,138.483357


In [8]:
# STEP 9 — Save CSV

file_name = "raw_data.csv"

if os.path.exists(file_name):

    print("CSV file already exists.")

else:

    df.to_csv(
        file_name,
        index=False
    )

CSV file already exists.


In [9]:
df["emissions_t"].describe()

count    710385.000000
mean          1.201308
std           6.428755
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max          54.834900
Name: emissions_t, dtype: float64

# 2. Data Integration and Materialisation/Caching

In [10]:
# Load Raw Data
df = pd.read_csv("raw_data.csv")

df.head()

,facility_name,facility_code,unit_code,power_mw,emissions_t,timestamp,lat,lon
0,Adelaide Desalination,ADP,ADPBA1,-0.005,0.0,2026-05-09T00:00:00+10:00,-35.100751,138.483357
1,Adelaide Desalination,ADP,ADPBA1,0.027,0.0,2026-05-09T00:05:00+10:00,-35.100751,138.483357
2,Adelaide Desalination,ADP,ADPBA1,0.008,0.0,2026-05-09T00:10:00+10:00,-35.100751,138.483357
3,Adelaide Desalination,ADP,ADPBA1,0.071,0.0,2026-05-09T00:15:00+10:00,-35.100751,138.483357
4,Adelaide Desalination,ADP,ADPBA1,0.009,0.0,2026-05-09T00:20:00+10:00,-35.100751,138.483357


In [11]:
# Basic Preprocessing
# Remove missing values
df = df.dropna()

# Remove duplicate rows
df = df.drop_duplicates()

# Convert timestamp datatype
df["timestamp"] = pd.to_datetime(
    df["timestamp"]
)

df.head()

,facility_name,facility_code,unit_code,power_mw,emissions_t,timestamp,lat,lon
0,Adelaide Desalination,ADP,ADPBA1,-0.005,0.0,2026-05-09 00:00:00+10:00,-35.100751,138.483357
1,Adelaide Desalination,ADP,ADPBA1,0.027,0.0,2026-05-09 00:05:00+10:00,-35.100751,138.483357
2,Adelaide Desalination,ADP,ADPBA1,0.008,0.0,2026-05-09 00:10:00+10:00,-35.100751,138.483357
3,Adelaide Desalination,ADP,ADPBA1,0.071,0.0,2026-05-09 00:15:00+10:00,-35.100751,138.483357
4,Adelaide Desalination,ADP,ADPBA1,0.009,0.0,2026-05-09 00:20:00+10:00,-35.100751,138.483357


In [12]:
# Sort Data by Time
df = df.sort_values(
    by="timestamp"
)

df = df.reset_index(drop=True)

df.head()

,facility_name,facility_code,unit_code,power_mw,emissions_t,timestamp,lat,lon
0,Adelaide Desalination,ADP,ADPBA1,-0.0050,0.0,2026-05-09 00:00:00+10:00,-35.100751,138.483357
1,Metz N,METZSF,METZSF1,0.0000,0.0,2026-05-09 00:00:00+10:00,-30.515484,151.859815
2,Melbourne A3,MREHA3,MREHA3,-4.0452,0.0,2026-05-09 00:00:00+10:00,-37.663819,144.726859
3,Melbourne A2,MREHA2,MREHA2,-1.5811,0.0,2026-05-09 00:00:00+10:00,-37.663934,144.726927
4,Melbourne A1,MREH,MREHA1,-1.4469,0.0,2026-05-09 00:00:00+10:00,-37.661157,144.726292


In [13]:
# Materialisation / Caching

file_name = "integrated_power_emissions_data.csv"

if os.path.exists(file_name):

    print("CSV file already exists.")

else:

    df.to_csv(
        file_name,
        index=False
    )

    print("Integrated CSV saved successfully.")

CSV file already exists.


# 3. Data Publishing via MQTT

In [1]:
!pip install paho-mqtt

In [2]:
import pandas as pd
import json
import time
from datetime import datetime
import paho.mqtt.client as mqtt

## 3.1 Load Integrated Dataset
This section loads the integrated power and emissions dataset generated from the previous data retrieval and materialisation steps. The dataset is used as the cached source for MQTT publishing, avoiding repeated external API calls during stream simulation.

In [3]:
# Load the integrated dataset generated from Task 2
df_stream = pd.read_csv("integrated_power_emissions_data.csv")

# Check dataset structure
print("Dataset shape:", df_stream.shape)
print(df_stream.head())
print(df_stream.dtypes)

Dataset shape: (710385, 8)
           facility_name facility_code unit_code  power_mw  emissions_t  \
0  Adelaide Desalination           ADP    ADPBA1   -0.0050          0.0   
1                 Metz N        METZSF   METZSF1    0.0000          0.0   
2           Melbourne A3        MREHA3    MREHA3   -4.0452          0.0   
3           Melbourne A2        MREHA2    MREHA2   -1.5811          0.0   
4           Melbourne A1          MREH    MREHA1   -1.4469          0.0   

                   timestamp        lat         lon  
0  2026-05-09 00:00:00+10:00 -35.100751  138.483357  
1  2026-05-09 00:00:00+10:00 -30.515484  151.859815  
2  2026-05-09 00:00:00+10:00 -37.663819  144.726859  
3  2026-05-09 00:00:00+10:00 -37.663934  144.726927  
4  2026-05-09 00:00:00+10:00 -37.661157  144.726292  
facility_name     object
facility_code     object
unit_code         object
power_mw         float64
emissions_t      float64
timestamp         object
lat              float64
lon              float6

## 3.2 Prepare MQTT Message Format
Before publishing the data as an MQTT stream, the records are sorted by their event timestamp. This ensures that messages are published in event-time order, which simulates an ordered electricity data stream.

In [4]:
# Convert timestamp to datetime format
df_stream["timestamp"] = pd.to_datetime(df_stream["timestamp"])

# Sort records by event time and facility/unit identifiers
df_stream = df_stream.sort_values(
    by=["timestamp", "facility_code", "unit_code"]
).reset_index(drop=True)

# Check the ordered data
print(df_stream[["timestamp", "facility_name", "facility_code", "unit_code", "power_mw", "emissions_t"]].head())
print(df_stream[["timestamp", "facility_name", "facility_code", "unit_code", "power_mw", "emissions_t"]].tail())

                  timestamp          facility_name facility_code unit_code  \
0 2026-05-09 00:00:00+10:00  Adelaide Desalination           ADP    ADPBA1   
1 2026-05-09 00:00:00+10:00                Hallett        AGLHAL    AGLHAL   
2 2026-05-09 00:00:00+10:00               Somerton        AGLSOM    AGLSOM   
3 2026-05-09 00:00:00+10:00                 Aldoga       ALDGASF  ALDGASF1   
4 2026-05-09 00:00:00+10:00               Angaston      ANGASTON   ANGAST1   

   power_mw  emissions_t  
0    -0.005          0.0  
1     0.000          0.0  
2     0.000          0.0  
3     0.000          0.0  
4     0.000          0.0  
                       timestamp facility_name facility_code unit_code  \
710380 2026-05-15 23:55:00+10:00     Yarranlea       YARANSF  YARANSF1   
710381 2026-05-15 23:55:00+10:00        Yarwun        YARWUN  YARWUN_1   
710382 2026-05-15 23:55:00+10:00       Yatpool        YATSF1    YATSF1   
710383 2026-05-15 23:55:00+10:00        Yendon      YENDONWF   YENDWF1   

## 3.3 MQTT Message Format
Each MQTT message represents one facility-level electricity generation event. The message includes facility information, event timestamp, power output, emissions, and location coordinates. The latitude and longitude fields support the downstream map-based dashboard.

In [5]:
def create_mqtt_message(row):
    """
    Convert one row of the integrated dataset into a JSON-compatible MQTT message.
    """
    message = {
        "facility_name": row["facility_name"],
        "facility_code": row["facility_code"],
        "unit_code": row["unit_code"],
        "timestamp": row["timestamp"].isoformat(),
        "power_mw": float(row["power_mw"]),
        "emissions_t": float(row["emissions_t"]),
        "lat": float(row["lat"]),
        "lon": float(row["lon"])
    }
    
    return message

In [6]:
# Test the message format using the first row
sample_message = create_mqtt_message(df_stream.iloc[0])

print(json.dumps(sample_message, indent=4))

{
    "facility_name": "Adelaide Desalination",
    "facility_code": "ADP",
    "unit_code": "ADPBA1",
    "timestamp": "2026-05-09T00:00:00+10:00",
    "power_mw": -0.005,
    "emissions_t": 0.0,
    "lat": -35.100751,
    "lon": 138.483357
}


## 3.4 MQTT Broker Configuration
A public MQTT broker is used to simulate the streaming pipeline. The publisher sends facility-level power and emissions messages to a shared MQTT topic. The downstream dashboard can subscribe to the same topic to receive real-time updates.

In [7]:
# MQTT broker configuration
MQTT_BROKER = "broker.hivemq.com"
MQTT_PORT = 1883
MQTT_TOPIC = "comp5339/A2group26/electricity/facility_stream"

print("MQTT Broker:", MQTT_BROKER)
print("MQTT Port:", MQTT_PORT)
print("MQTT Topic:", MQTT_TOPIC)

MQTT Broker: broker.hivemq.com
MQTT Port: 1883
MQTT Topic: comp5339/A2group26/electricity/facility_stream


In [8]:
def connect_mqtt():
    """
    Create and connect an MQTT client.
    """
    client = mqtt.Client(client_id="comp5339_mqtt_publisher")
    
    client.connect(
        MQTT_BROKER,
        MQTT_PORT,
        keepalive=60
    )
    
    return client

In [9]:
client = connect_mqtt()
client.loop_start()

print("Connected to MQTT broker successfully.")

client.loop_stop()
client.disconnect()

/var/folders/56/5hhy_nc96sd14853bqkqxlg00000gn/T/ipykernel_44764/2809283726.py:5: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client(client_id="comp5339_mqtt_publisher")


Connected to MQTT broker successfully.


<MQTTErrorCode.MQTT_ERR_SUCCESS: 0>

## 3.5 Publish MQTT Messages

The following function publishes each row of the integrated dataset as a JSON message to the MQTT topic. A delay of 0.1 seconds is added between messages to simulate streaming behaviour rather than sending all records at once.

In [10]:
def publish_single_message(client, row):
    """
    Publish one row as a JSON MQTT message.
    """
    message = create_mqtt_message(row)
    payload = json.dumps(message)
    
    result = client.publish(
        MQTT_TOPIC,
        payload=payload,
        qos=0
    )
    
    if result.rc != mqtt.MQTT_ERR_SUCCESS:
        print(f"Failed to publish message. MQTT result code: {result.rc}")
    
    return result

In [11]:
def publish_dataframe(client, df, delay_seconds=0.1, max_messages=None):
    """
    Publish the integrated dataset row by row in event-time order.
    
    Parameters:
    client: MQTT client
    df: sorted DataFrame
    delay_seconds: delay between each MQTT message
    max_messages: optional limit for testing/demo
    """
    publish_count = 0
    
    if max_messages is not None:
        df_to_publish = df.head(max_messages)
    else:
        df_to_publish = df
    
    for _, row in df_to_publish.iterrows():
        result = publish_single_message(client, row)
        publish_count += 1
        
        print(
            f"Published {publish_count}: "
            f"{row['timestamp']} | "
            f"{row['facility_name']} | "
            f"Power={row['power_mw']} MW | "
            f"Emissions={row['emissions_t']} t"
        )
        
        time.sleep(delay_seconds)
    
    print(f"Finished publishing {publish_count} messages.")

In [12]:
def on_message(client, userdata, message):
    """
    Callback function for receiving MQTT messages.
    """
    payload = message.payload.decode("utf-8")
    print("Received message:")
    print(payload)


subscriber = mqtt.Client(client_id="comp5339_test_subscriber")
subscriber.on_message = on_message

subscriber.connect(MQTT_BROKER, MQTT_PORT, keepalive=60)
subscriber.subscribe(MQTT_TOPIC)

subscriber.loop_start()

print(f"Subscribed to topic: {MQTT_TOPIC}")

/var/folders/56/5hhy_nc96sd14853bqkqxlg00000gn/T/ipykernel_44764/976399684.py:10: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  subscriber = mqtt.Client(client_id="comp5339_test_subscriber")


Subscribed to topic: comp5339/A2group26/electricity/facility_stream
Received message:
{"facility_name": "Bastyan", "facility_code": "BASTYAN", "unit_code": "BASTYAN", "timestamp": "2026-05-09T00:00:00+10:00", "power_mw": 79.59, "emissions_t": 0.0, "lat": -41.736045, "lon": 145.532064}
Received message:
{"facility_name": "Bayswater", "facility_code": "BAYSW", "unit_code": "BW01", "timestamp": "2026-05-09T00:00:00+10:00", "power_mw": 659.0555, "emissions_t": 48.9952, "lat": -32.394981, "lon": 150.94949}
Received message:
{"facility_name": "Bouldercombe", "facility_code": "BBATTERY", "unit_code": "BBATTERY1", "timestamp": "2026-05-09T00:00:00+10:00", "power_mw": 0.0, "emissions_t": 0.0, "lat": -23.535251, "lon": 150.489057}
Received message:
{"facility_name": "Bell Bay Three", "facility_code": "BBP31", "unit_code": "BBTHREE1", "timestamp": "2026-05-09T00:00:00+10:00", "power_mw": 0.0, "emissions_t": 0.0, "lat": -41.139324, "lon": 146.905522}
Received message:
{"facility_name": "Beryl", "f

In [13]:
client = connect_mqtt()
client.loop_start()

publish_dataframe(
    client=client,
    df=df_stream,
    delay_seconds=0.1,
    max_messages=20
)

client.loop_stop()
client.disconnect()

/var/folders/56/5hhy_nc96sd14853bqkqxlg00000gn/T/ipykernel_44764/2809283726.py:5: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client(client_id="comp5339_mqtt_publisher")


Published 1: 2026-05-09 00:00:00+10:00 | Adelaide Desalination | Power=-0.005 MW | Emissions=0.0 t
Published 2: 2026-05-09 00:00:00+10:00 | Hallett | Power=0.0 MW | Emissions=0.0 t
Published 3: 2026-05-09 00:00:00+10:00 | Somerton | Power=0.0 MW | Emissions=0.0 t
Published 4: 2026-05-09 00:00:00+10:00 | Aldoga | Power=0.0 MW | Emissions=0.0 t
Published 5: 2026-05-09 00:00:00+10:00 | Angaston | Power=0.0 MW | Emissions=0.0 t
Published 6: 2026-05-09 00:00:00+10:00 | Ararat | Power=55.5 MW | Emissions=0.0 t
Published 7: 2026-05-09 00:00:00+10:00 | Avonlie | Power=0.0 MW | Emissions=0.0 t
Published 8: 2026-05-09 00:00:00+10:00 | Braemar 2 | Power=0.1913 MW | Emissions=0.0092 t
Published 9: 2026-05-09 00:00:00+10:00 | Ballarat | Power=0.0 MW | Emissions=0.0 t
Published 10: 2026-05-09 00:00:00+10:00 | Bango | Power=40.9992 MW | Emissions=0.0 t
Published 11: 2026-05-09 00:00:00+10:00 | Bannerton | Power=0.0 MW | Emissions=0.0 t
Published 12: 2026-05-09 00:00:00+10:00 | Barcaldine | Power=0.0 

<MQTTErrorCode.MQTT_ERR_SUCCESS: 0>

In [14]:
subscriber.loop_stop()
subscriber.disconnect()

print("Subscriber stopped and disconnected.")

Subscriber stopped and disconnected.


# 4. Schema from Assignment 1
We should implement the schema (which was designed in Assignment 1) in the database, and get the DuckDB  database schema.

In addition to the code in the code block below, our assignment folder also contains a file named `task4_build_database.py`. You can also run the command `python task4_build_database.py` directly in the terminal to generate `assignment2_energy_stream.duckdb`.

In [15]:
import duckdb
import pandas as pd
from pathlib import Path

CSV_PATH = "integrated_power_emissions_data.csv"
DB_PATH = "assignment2_energy_stream.duckdb"


def build_database(csv_path=CSV_PATH, db_path=DB_PATH):
    if not Path(csv_path).exists():
        raise FileNotFoundError(f"Cannot find {csv_path}")

    con = duckdb.connect(db_path)

    con.execute("INSTALL spatial;")
    con.execute("LOAD spatial;")

    con.execute(f"""
    CREATE OR REPLACE TABLE raw_power_emissions AS
    SELECT *
    FROM read_csv_auto('{csv_path}', header=True);
    """)

    con.execute("""
    CREATE OR REPLACE TABLE facilities AS
    SELECT DISTINCT
        TRIM(facility_code) AS facility_code,
        TRIM(facility_name) AS facility_name,
        CAST(lat AS DOUBLE) AS latitude,
        CAST(lon AS DOUBLE) AS longitude,
        ST_GeomFromText('POINT(' || CAST(lon AS VARCHAR) || ' ' || CAST(lat AS VARCHAR) || ')') AS geometry
    FROM raw_power_emissions
    WHERE facility_code IS NOT NULL
      AND facility_name IS NOT NULL
      AND lat IS NOT NULL
      AND lon IS NOT NULL;
    """)

    con.execute("""
    CREATE OR REPLACE TABLE facility_units AS
    SELECT DISTINCT
        TRIM(unit_code) AS unit_code,
        TRIM(facility_code) AS facility_code
    FROM raw_power_emissions
    WHERE unit_code IS NOT NULL
      AND facility_code IS NOT NULL;
    """)

    con.execute("""
    CREATE OR REPLACE TABLE power_emissions_measurements AS
    SELECT
        CAST(timestamp AS TIMESTAMP) AS event_time,
        TRIM(facility_code) AS facility_code,
        TRIM(unit_code) AS unit_code,
        CAST(power_mw AS DOUBLE) AS power_mw,
        CAST(emissions_t AS DOUBLE) AS emissions_t
    FROM raw_power_emissions
    WHERE timestamp IS NOT NULL
      AND facility_code IS NOT NULL
      AND unit_code IS NOT NULL;
    """)

    con.execute("""
    CREATE OR REPLACE TABLE mqtt_received_measurements (
        received_at TIMESTAMP DEFAULT current_timestamp,
        event_time TIMESTAMP,
        facility_name VARCHAR,
        facility_code VARCHAR,
        unit_code VARCHAR,
        power_mw DOUBLE,
        emissions_t DOUBLE,
        latitude DOUBLE,
        longitude DOUBLE,
        geometry GEOMETRY
    );
    """)

    con.execute("""
    CREATE OR REPLACE VIEW latest_facility_status AS
    WITH latest AS (
        SELECT
            facility_code,
            MAX(event_time) AS latest_time
        FROM mqtt_received_measurements
        GROUP BY facility_code
    )
    SELECT
        m.facility_name,
        m.facility_code,
        m.unit_code,
        m.event_time,
        m.power_mw,
        m.emissions_t,
        m.latitude,
        m.longitude,
        m.geometry
    FROM mqtt_received_measurements m
    JOIN latest l
      ON m.facility_code = l.facility_code
     AND m.event_time = l.latest_time;
    """)

    print("Database created:", db_path)

    for table in ["facilities", "facility_units", "power_emissions_measurements"]:
        n = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        print(f"{table}: {n} rows")

    con.close()


if __name__ == "__main__":
    build_database()

Database created: assignment2_energy_stream.duckdb
facilities: 353 rows
facility_units: 353 rows
power_emissions_measurements: 710385 rows


# 5. Data Subscription and Visualisation

For this section, we created a separate file named `task5_dashboard_app.py` to implement the functionality.

Follow the steps below to reproduce our visualization implementation process:

Step 1: Open the Terminal, drag the entire project folder into the Terminal window to ensure you’re in the project directory. Then, enter the command `brew install mosquitto` to install Mosquitto locally. Next, use the command `mosquitto -v` to start the local broker and keep it running.

Step 2: Open a new terminal window, make sure you are in the project directory, and run the command `python mqtt_streaming.py`. The terminal will then display the published data.

Step 3: Open a third terminal, make sure you're in the project directory, and run the command `streamlit run task5_dashboard_app.py`. This will automatically open your browser and take you to the dashboard.


Please note that we have included three distinct sections of code in `mqtt_streaming.py`:
1. Demo mode: This mode publishes only 30 messages and then automatically terminates, allowing you to quickly verify that the code is working correctly.

In the submitted assignment’s .py file, this section of code is commented out. If you choose to reproduce this code, please start the dashboard first, then start the publisher. Otherwise, the dashboard will not be subscribed to receive messages when the publisher publishes them, and data will not be received successfully. The dashboard page will also display the message `No MQTT messages received yet.`

2. Code that meets the “Continuous Execution” requirement: The publisher sends 300 messages at 0.5-second intervals, waits 60 seconds, and then continues with the next round of publishing. This provides ample time to observe dynamic changes in the dashboard. To stop the process, use the `Control + C` command in the terminal.

Currently, this is the code segment enabled in the `mqtt_streaming.py` file, and the dashboard screenshot in the report is the result of running this code.

3. Code that strictly requires all records to be published: Publish all 710,385 records, with an interval of 0.1 seconds between each.

A full round of publishing takes approximately 19.7 hours. For timeliness reasons, this code is also commented out in the .py file.



Overall, for Task 5, we developed a Streamlit-based map dashboard. The dashboard subscribes to the MQTT topic published by the streaming module and inserts each received message into the DuckDB database. The latest record for each facility is then visualised on an interactive map. Each marker shows the facility name and the selected live metric, either power output or CO2 emissions. Clicking a marker displays a popup containing the facility name, unit code, timestamp, latest power output, and emissions.

# 6. Continuous Execution
To simulate an unbounded data stream, the MQTT publishing process is wrapped in a continuous loop. In each cycle, the program loads the cached integrated dataset, sorts it by event time, publishes MQTT messages with a 0.1-second delay between records, and then waits 60 seconds before starting the next cycle.

In [16]:
def load_and_prepare_stream_data(csv_path="integrated_power_emissions_data.csv"):
    """
    Load the cached integrated dataset and prepare it for streaming.
    """
    df = pd.read_csv(csv_path)
    
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    
    df = df.dropna(
        subset=[
            "facility_name",
            "facility_code",
            "unit_code",
            "timestamp",
            "power_mw",
            "emissions_t",
            "lat",
            "lon"
        ]
    )
    
    df = df.sort_values(
        by=["timestamp", "facility_code", "unit_code"]
    ).reset_index(drop=True)
    
    return df

In [17]:
def run_continuous_streaming(
    csv_path="integrated_power_emissions_data.csv",
    delay_between_messages=0.1,
    delay_between_cycles=60,
    max_messages_per_cycle=None
):
    """
    Run the MQTT publisher continuously to simulate an unbounded data stream.
    
    Each cycle:
    1. Loads the cached integrated dataset
    2. Sorts records by event time
    3. Publishes messages one by one through MQTT
    4. Waits 60 seconds before the next cycle
    """
    client = connect_mqtt()
    client.loop_start()
    
    cycle_count = 0
    
    try:
        while True:
            cycle_count += 1
            
            print("=" * 80)
            print(f"Starting streaming cycle {cycle_count}")
            print("Loading integrated dataset...")
            
            df = load_and_prepare_stream_data(csv_path)
            
            print(f"Prepared {len(df)} records for MQTT publishing.")
            print("Publishing messages...")
            
            publish_dataframe(
                client=client,
                df=df,
                delay_seconds=delay_between_messages,
                max_messages=max_messages_per_cycle
            )
            
            print(f"Cycle {cycle_count} completed.")
            print(f"Waiting {delay_between_cycles} seconds before the next cycle...")
            
            time.sleep(delay_between_cycles)
    
    except KeyboardInterrupt:
        print("Continuous streaming stopped by user.")
    
    finally:
        client.loop_stop()
        client.disconnect()
        print("MQTT client disconnected.")

In [18]:
def run_one_streaming_cycle(
    csv_path="integrated_power_emissions_data.csv",
    delay_between_messages=0.1,
    max_messages_per_cycle=30
):
    """
    Run one MQTT publishing cycle for notebook testing/demo.

    This function publishes a limited number of records and then stops,
    so the notebook can be reviewed without entering an infinite loop.
    """
    print("=" * 80)
    print("Starting one-cycle MQTT streaming test")
    print("Loading integrated dataset...")

    df = load_and_prepare_stream_data(csv_path)

    print(f"Prepared {len(df)} records for MQTT publishing.")
    print(f"Publishing up to {max_messages_per_cycle} messages...")

    client = connect_mqtt()
    client.loop_start()

    try:
        publish_dataframe(
            client=client,
            df=df,
            delay_seconds=delay_between_messages,
            max_messages=max_messages_per_cycle
        )

        print("One-cycle MQTT streaming test completed.")

    finally:
        client.loop_stop()
        client.disconnect()
        print("MQTT client disconnected.")


In [19]:
# Demo / optional continuous streaming example
# This is intentionally commented out because run_continuous_streaming() keeps looping.
# Use run_one_streaming_cycle() below for the notebook test run instead.

# run_continuous_streaming(
#     csv_path="integrated_power_emissions_data.csv",
#     delay_between_messages=0.1,
#     delay_between_cycles=60,
#     max_messages_per_cycle=30
# )


For Task 3, the integrated power and emissions dataset was published as an MQTT data stream. Each record was converted into a JSON message containing the facility name, facility code, unit code, timestamp, power output, emissions, and facility coordinates. Before publishing, all records were sorted by event timestamp, facility code, and unit code to ensure event-time ordering. The publisher sent each JSON message to the MQTT topic `comp5339/A2group26/electricity/facility_stream`, with a 0.1-second delay between messages to simulate a real-time ordered stream.

For Task 6, the MQTT publishing process was wrapped in a continuous execution loop. In each cycle, the cached integrated CSV file was loaded, cleaned, sorted, and published through MQTT. After each publishing cycle, the program waited 60 seconds before starting the next cycle. This design simulates an unbounded data stream while avoiding unnecessary repeated API calls during testing.

## Deliverable Files for MQTT Streaming
The MQTT streaming and continuous execution logic was also exported into a standalone Python script named `mqtt_streaming.py`. This script contains the MQTT broker configuration, message formatting function, event-time ordered publishing function, one-cycle demo function, and continuous execution loop.

The project also includes a `requirements.txt` file listing the Python packages required to run the data retrieval and MQTT streaming components in a clean virtual environment.

## MQTT Streaming Publisher

This section demonstrates the MQTT streaming publisher used to simulate real-time data ingestion.

For testing and marking purposes, only 30 messages are published below.

Publishing the full dataset of 710,385 records with a 0.1-second delay between messages would take approximately 19.7 hours per cycle, so the full streaming code is provided but kept commented out.

In [20]:
# =========================
# Test Run: Publish 30 Messages
# =========================
# This test run publishes only 30 records to verify that the MQTT streaming
# workflow can load the dataset, connect to the broker, and publish messages.

run_one_streaming_cycle(
    csv_path="integrated_power_emissions_data.csv",
    delay_between_messages=0.1,
    max_messages_per_cycle=30
)

Starting one-cycle MQTT streaming test
Loading integrated dataset...
Prepared 710385 records for MQTT publishing.
Publishing up to 30 messages...


/var/folders/56/5hhy_nc96sd14853bqkqxlg00000gn/T/ipykernel_44764/2809283726.py:5: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client(client_id="comp5339_mqtt_publisher")


Published 1: 2026-05-09 00:00:00+10:00 | Adelaide Desalination | Power=-0.005 MW | Emissions=0.0 t
Published 2: 2026-05-09 00:00:00+10:00 | Hallett | Power=0.0 MW | Emissions=0.0 t
Published 3: 2026-05-09 00:00:00+10:00 | Somerton | Power=0.0 MW | Emissions=0.0 t
Published 4: 2026-05-09 00:00:00+10:00 | Aldoga | Power=0.0 MW | Emissions=0.0 t
Published 5: 2026-05-09 00:00:00+10:00 | Angaston | Power=0.0 MW | Emissions=0.0 t
Published 6: 2026-05-09 00:00:00+10:00 | Ararat | Power=55.5 MW | Emissions=0.0 t
Published 7: 2026-05-09 00:00:00+10:00 | Avonlie | Power=0.0 MW | Emissions=0.0 t
Published 8: 2026-05-09 00:00:00+10:00 | Braemar 2 | Power=0.1913 MW | Emissions=0.0092 t
Published 9: 2026-05-09 00:00:00+10:00 | Ballarat | Power=0.0 MW | Emissions=0.0 t
Published 10: 2026-05-09 00:00:00+10:00 | Bango | Power=40.9992 MW | Emissions=0.0 t
Published 11: 2026-05-09 00:00:00+10:00 | Bannerton | Power=0.0 MW | Emissions=0.0 t
Published 12: 2026-05-09 00:00:00+10:00 | Barcaldine | Power=0.0 

In [21]:
# Optional: continuous streaming mode
# Uncomment the code below to run repeated streaming cycles.

# run_continuous_streaming(
#     csv_path="integrated_power_emissions_data.csv",
#     delay_between_messages=0.1,
#     delay_between_cycles=60,
#     max_messages_per_cycle=300
# )

In [22]:
# Optional: full dataset streaming
# Publishing all 710,385 records with a 0.1-second delay takes approximately
# 19.7 hours per cycle.
#
# This is kept commented out to avoid long execution time during notebook review.

# run_continuous_streaming(
#     csv_path="integrated_power_emissions_data.csv",
#     delay_between_messages=0.1,
#     delay_between_cycles=60,
#     max_messages_per_cycle=None
# )